# Module 07: ChipWhisperer-Lite Hardware & First Lab

This lab simulates the ChipWhisperer workflow for power trace acquisition.

**Objectives:**
1. Simulate scope configuration and trace capture
2. Visualize single power traces
3. Overlay multiple traces to observe data-dependent patterns
4. Compute mean and standard deviation of traces

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("CHIPWHISPERER SCOPE SIMULATION")
print("=" * 60)

# Simulate ChipWhisperer scope configuration
class SimulatedScope:
    def __init__(self):
        self.clkgen_freq = 7.37e6  # Hz
        self.adc_mul = 4
        self.gain = 45  # dB
        self.adc_bits = 10
        self.sample_rate = self.clkgen_freq * self.adc_mul
        self.postsamples = 24000
        
    def print_config(self):
        print(f"Clock frequency: {self.clkgen_freq/1e6:.2f} MHz")
        print(f"ADC multiplier: {self.adc_mul}")
        print(f"Sample rate: {self.sample_rate/1e6:.2f} MS/s")
        print(f"Gain: {self.gain} dB")
        print(f"ADC bits: {self.adc_bits}")
        print(f"Post-trigger samples: {self.postsamples}")
        
    def capture_trace(self, plaintext):
        """Simulate capturing a power trace for one AES encryption"""
        np.random.seed(hash(plaintext.tobytes()) % 2**31)
        
        # Simulate AES power trace with 10 rounds
        trace = np.zeros(self.postsamples)
        
        # Initial AddRoundKey (sample 0-200)
        trace[100:200] += np.random.normal(0.5, 0.1, 100)
        
        # 10 AES rounds (each ~2000 samples)
        for round_num in range(10):
            start = 500 + round_num * 2200
            
            # SubBytes (high peaks)
            for i in range(16):
                sbox_in = plaintext[i] ^ (round_num * 0x10 + i)
                sbox_out = [0x63,0x7C,0x77,0x7B,0xF2,0x6B,0x6F,0xC5,0x30,0x01,0x67,0x2B,0xFE,0xD7,0xAB,0x76,
                           0xCA,0x82,0xC9,0x7D,0xFA,0x59,0x47,0xF0,0xAD,0xD4,0xA2,0xAF,0x9C,0xA4,0x72,0xC0,
                           0xB7,0xFD,0x93,0x26,0x36,0x3F,0xF7,0xCC,0x34,0xA5,0xE5,0xF1,0x71,0xD8,0x31,0x15,
                           0x04,0xC7,0x23,0xC3,0x18,0x96,0x05,0x9A,0x07,0x12,0x80,0xE2,0xEB,0x27,0xB2,0x75,
                           0x09,0x83,0x2C,0x1A,0x1B,0x6E,0x5A,0xA0,0x52,0x3B,0xD6,0xB3,0x29,0xE3,0x2F,0x84,
                           0x53,0xD1,0x00,0xED,0x20,0xFC,0xB1,0x5B,0x6A,0xCB,0xBE,0x39,0x4A,0x4C,0x58,0xCF,
                           0xD0,0xEF,0xAA,0xFB,0x43,0x4D,0x33,0x85,0x45,0xF9,0x02,0x7F,0x50,0x3C,0x9F,0xA8,
                           0x51,0xA3,0x40,0x8F,0x92,0x9D,0x38,0xF5,0xBC,0xB6,0xDA,0x21,0x10,0xFF,0xF3,0xD2,
                           0xCD,0x0C,0x13,0xEC,0x5F,0x97,0x44,0x17,0xC4,0xA7,0x7E,0x3D,0x64,0x5D,0x19,0x73,
                           0x60,0x81,0x4F,0xDC,0x22,0x2A,0x90,0x88,0x46,0xEE,0xB8,0x14,0xDE,0x5E,0x0B,0xDB,
                           0xE0,0x32,0x3A,0x0A,0x49,0x06,0x24,0x5C,0xC2,0xD3,0xAC,0x62,0x91,0x95,0xE4,0x79,
                           0xE7,0xC8,0x37,0x6D,0x8D,0xD5,0x4E,0xA9,0x6C,0x56,0xF4,0xEA,0x65,0x7A,0xAE,0x08,
                           0xBA,0x78,0x25,0x2E,0x1C,0xA6,0xB4,0xC6,0xE8,0xDD,0x74,0x1F,0x4B,0xBD,0x8B,0x8A,
                           0x70,0x3E,0xB5,0x66,0x48,0x03,0xF6,0x0E,0x61,0x35,0x57,0xB9,0x86,0xC1,0x1D,0x9E,
                           0xE1,0xF8,0x98,0x11,0x69,0xD9,0x8E,0x94,0x9B,0x1E,0x87,0xE9,0xCE,0x55,0x28,0xDf,
                           0x8C,0xA1,0x89,0x0D,0xBF,0xE6,0x42,0x68,0x41,0x99,0x2D,0x0F,0xB0,0x54,0xBB,0x16]
                hw = bin(sbox_out[sbox_in % 256]).count('1')
                peak_pos = start + i * 12
                if peak_pos < self.postsamples:
                    trace[peak_pos] += hw * 0.3
            
            # Add round noise
            end = start + 2000
            trace[start:end] += np.random.normal(0.2, 0.05, min(2000, end-start))
        
        # Add overall noise
        trace += np.random.normal(0, 0.1, self.postsamples)
        
        return trace

# Create simulated scope
scope = SimulatedScope()
scope.print_config()

# Capture a single trace
plaintext = np.random.randint(0, 256, 16, dtype=np.uint8)
trace = scope.capture_trace(plaintext)

print(f"\nCaptured trace: {len(trace)} samples")
print(f"Plaintext: {plaintext.tobytes().hex()}")

# Plot single trace
plt.figure(figsize=(14, 5))
plt.plot(trace, linewidth=0.5)
plt.title('Single Power Trace (AES Encryption)')
plt.xlabel('Sample')
plt.ylabel('Power (ADC units)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Capture and overlay multiple traces
print("=" * 60)
print("MULTIPLE TRACES: DATA-DEPENDENT PATTERNS")
print("=" * 60)

n_traces = 20
traces = []
plaintexts = []

for i in range(n_traces):
    pt = np.random.randint(0, 256, 16, dtype=np.uint8)
    traces.append(scope.capture_trace(pt))
    plaintexts.append(pt)

traces = np.array(traces)

# Plot overlay of all traces
plt.figure(figsize=(14, 6))
for i, t in enumerate(traces):
    plt.plot(t, alpha=0.3, linewidth=0.3)
plt.title(f'Overlay of {n_traces} Power Traces')
plt.xlabel('Sample')
plt.ylabel('Power')
plt.tight_layout()
plt.show()

# Compute and plot mean trace
mean_trace = np.mean(traces, axis=0)
std_trace = np.std(traces, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(mean_trace, linewidth=0.5, color='blue')
axes[0].set_title('Mean Power Trace')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Mean Power')
axes[0].grid(True, alpha=0.3)

axes[1].plot(std_trace, linewidth=0.5, color='red')
axes[1].set_title('Standard Deviation (Data-Dependent Variation)')
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Std Dev')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean trace length: {len(mean_trace)} samples")
print(f"Max std dev: {np.max(std_trace):.4f}")
print(f"Points with high variation: {np.sum(std_trace > 0.1)} samples")

In [ ]:
# Identify Points of Interest (POI) via t-test
print("=" * 60)
print("POINTS OF INTEREST (POI) IDENTIFICATION")
print("=" * 60)

# Simple TVLA: compare fixed vs random plaintexts
n_tvla = 100

# Fixed set: all zeros
fixed_traces = np.array([scope.capture_trace(np.zeros(16, dtype=np.uint8)) for _ in range(n_tvla)])

# Random set
random_traces = np.array([scope.capture_trace(np.random.randint(0, 256, 16, dtype=np.uint8)) for _ in range(n_tvla)])

# Welch's t-test
t_values = np.zeros(scope.postsamples)
for t in range(scope.postsamples):
    mean_f = np.mean(fixed_traces[:, t])
    mean_r = np.mean(random_traces[:, t])
    var_f = np.var(fixed_traces[:, t], ddof=1)
    var_r = np.var(random_traces[:, t], ddof=1)
    se = np.sqrt(var_f/n_tvla + var_r/n_tvla)
    if se > 0:
        t_values[t] = (mean_f - mean_r) / se

# Find top POI
threshold = 4.5
poi_indices = np.where(np.abs(t_values) > threshold)[0]
print(f"Leaking samples (|t| > {threshold}): {len(poi_indices)}")

if len(poi_indices) > 0:
    # Cluster nearby POIs
    clusters = []
    current_cluster = [poi_indices[0]]
    for i in range(1, len(poi_indices)):
        if poi_indices[i] - poi_indices[i-1] < 50:  # Within 50 samples
            current_cluster.append(poi_indices[i])
        else:
            clusters.append(current_cluster)
            current_cluster = [poi_indices[i]]
    clusters.append(current_cluster)
    
    print(f"\nIdentified {len(clusters)} POI clusters:")
    for i, cluster in enumerate(clusters[:10]):
        center = np.mean(cluster)
        max_t = np.max(np.abs(t_values[cluster]))
        print(f"  Cluster {i+1}: samples {cluster[0]}-{cluster[-1]}, center={center:.0f}, max|t|={max_t:.1f}")

# Plot t-test with POIs marked
plt.figure(figsize=(14, 5))
plt.plot(np.abs(t_values), linewidth=0.5)
plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({threshold})')
if len(poi_indices) > 0:
    plt.scatter(poi_indices, np.abs(t_values[poi_indices]), c='red', s=10, zorder=5, label='POI')
plt.title('TVLA t-test with Points of Interest')
plt.xlabel('Sample')
plt.ylabel('|t|')
plt.legend()
plt.tight_layout()
plt.show()